# DỰ ĐOÁN GIÁ NHÀ VN — DEEP LEARNING (PYTORCH)
**Bài toán:** Hồi quy (Regression)
---
**Mô hình:** Tương tự Deep Learning MLP.

In [ ]:
import numpy as np
import pandas as pd
import joblib, os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'preprocessed_data.npz'))
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']

X_tr_t = torch.tensor(X_train, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=256, shuffle=True)


In [ ]:
class RegressorMLP(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = RegressorMLP(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(1, 21):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0: print(f'Epoch {epoch}: Loss = {loss.item():.4f}')


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_te_t).numpy().ravel()
preds = np.clip(preds, 0, 15)

y_test_real = np.expm1(y_test)
preds_real = np.expm1(preds)
mae = mean_absolute_error(y_test_real, preds_real)
rmse = np.sqrt(mean_squared_error(y_test_real, preds_real))
r2 = r2_score(y_test_real, preds_real)
print(f"Deep Learning -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")

torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'hp_mlp_best.pth'))
np.savez_compressed(os.path.join(MODEL_DIR, 'hp_dl_preds.npz'), preds_real=preds_real)
print('Saved PyTorch Model.')
